# 🌿 Plant Explorer
## A Middle School Plant Science Investigation

In this notebook you will:
- 📸 Load a **mystery plant** photo and photos of **known plants**
- 🧠 Use **CLIP** (an AI vision model) to compare them without any internet API
- 📊 Get **similarity scores** showing how closely each plant matches
- 🔵 See plant embeddings **plotted in 2D space**
- 🗺️ **Map** where each plant was found

> **No API key needed — everything runs on this computer!**
>
> **You only need to change a few lines — look for the ✏️ symbol.**


---
## 🧠 How does CLIP work?

CLIP (**C**ontrastive **L**anguage-**I**mage **P**retraining) was trained on
400 million photos to learn what makes images similar or different.

It converts every photo into a list of **512 numbers** called an **embedding**
— think of it as a GPS address in a 512-dimensional space.
Similar-looking plants end up at nearby addresses.

We measure similarity using **cosine similarity**: the cosine of the angle
between two embedding vectors.

| Cosine similarity | What it means |
|---|---|
| 1.0 (angle = 0°) | Identical — vectors point the same way |
| 0.85 (angle ≈ 30°) | Very similar |
| 0.70 (angle ≈ 46°) | Quite different |
| 0.60 (angle ≈ 54°) | Very different |

Raw CLIP scores sit between ~0.60 and 1.00 for real photos, so we
**rescale** them to a 0–100% display score for easier reading.


---
## ⚙️ Setup — Run this cell first


In [ ]:
import subprocess, sys

packages = ["transformers", "torch", "Pillow", "folium", "scikit-learn"]
for pkg in packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

print("✅ All packages installed!")


In [ ]:
# Load our plant science helper functions
from plant_identifier import (
    check_photos, load_clip, show_image, show_all_plants,
    extract_gps, compare_plants,
    plot_results, plot_embeddings, make_map
)

# Load the CLIP model (downloads ~600 MB the first time, then cached)
load_clip()


---
## ✏️ Step 1 — Load your plant photos

**How to upload photos to JupyterHub:**
1. Open the **file browser** panel on the left
2. Click the **Upload** button (↑ arrow)
3. Select your photos — they appear in the file list

Then update the file names below to match your photos.

> 💡 **Tip:** Take photos outside with your phone, or download from
> [Wikipedia](https://en.wikipedia.org) or
> [iNaturalist](https://www.inaturalist.org).
> For best results, fill the frame with the plant.


In [ ]:
# ✏️ Replace these with YOUR photo file names
mystery_photo = "mystery_plant.jpg"

known_plants = [
    {"name": "Dandelion",    "path": "dandelion.jpg"},
    {"name": "White Clover", "path": "clover.jpg"},
    {"name": "Plantain",     "path": "plantain.jpg"},   # the plant, not the banana!
]
# Add more known plants by copying one of the lines above

# ✏️ If your photos don't have GPS built in, enter coordinates manually.
# Find coordinates: go to maps.google.com, right-click any spot → "What's here?"
manual_coords = {
    "Mystery Plant": {"latitude": 39.9526, "longitude": -75.1652},
    "Dandelion":     {"latitude": 39.9530, "longitude": -75.1648},
    "White Clover":  {"latitude": 39.9528, "longitude": -75.1655},
    "Plantain":      {"latitude": 39.9522, "longitude": -75.1660},
}

print(f"Mystery plant : {mystery_photo}")
print(f"Known plants  : {[p['name'] for p in known_plants]}")


In [ ]:
# Verify all photo files exist before going further
check_photos(mystery_photo, known_plants)


### 📷 Display all photos


In [ ]:
show_all_plants(mystery_photo, known_plants)


---
## 📍 Step 2 — Find where each photo was taken

Modern smartphones often save **GPS coordinates** inside the photo file
as hidden *EXIF metadata*. We'll read that automatically. If a photo
doesn't have GPS data (common with downloaded photos) we use the
coordinates you entered above.


In [ ]:
all_photos = [{"name": "Mystery Plant", "path": mystery_photo}] + \
             [{"name": p["name"], "path": p["path"]} for p in known_plants]

plant_locations = []

print("Checking each photo for GPS data...\n")
for photo in all_photos:
    name = photo["name"]
    gps  = extract_gps(photo["path"])

    if gps:
        source = "EXIF (from photo)"
    elif name in manual_coords:
        gps    = manual_coords[name]
        source = "manual coordinates"
    else:
        source = None

    if gps:
        print(f"  ✅ {name:20s} → {gps['latitude']:.5f}, {gps['longitude']:.5f}  ({source})")
        plant_locations.append({"name": name, **gps})
    else:
        print(f"  ⚠️  {name:20s} → no GPS (add to manual_coords above)")

print(f"\n📍 {len(plant_locations)} location(s) ready to map")


---
## 🤖 Step 3 — Compare the mystery plant to known plants

CLIP converts every photo into a **512-number embedding vector**.
We then measure the **cosine similarity** between the mystery plant
vector and each known plant vector.

⏳ Comparison is fast once the model is loaded — usually under 5 seconds.


In [ ]:
result = compare_plants(mystery_photo, known_plants)


### 📊 Similarity bar chart


In [ ]:
plot_results(result)


---
## 🔵 Step 4 — Visualise the embedding space

Each plant is a point in 512-dimensional space — impossible to draw directly.
We use **PCA** (Principal Component Analysis) to squash 512 dimensions
down to 2 so we can plot them.

**Plants that are close together on this plot are visually similar
according to CLIP.** The mystery plant (red) should appear nearest
to the known plant it most resembles.


In [ ]:
plot_embeddings(mystery_photo, known_plants)


---
## 🗺️ Step 5 — Map the plant locations

Click any marker to see the plant name and coordinates.
Zoom in/out with the scroll wheel or + / − buttons.


In [ ]:
plant_map = make_map(plant_locations, zoom_start=15)
plant_map


---
## 💬 Step 6 — Think like a scientist!

Answer these in your science notebook or type in the cell below.

**1. Which known plant had the highest similarity score?**
   Do you think CLIP made the right choice? Why or why not?

**2. Look at the embedding plot (Step 4).**
   Is the mystery plant (red dot) near the plant with the highest score?
   What does it mean when two points are far apart?

**3. Cosine similarity vs raw percentage:**
   The raw cosine values printed in the bar chart are all between 0.60 and 1.00.
   Why doesn't CLIP ever give a score of 0.0 for two completely different plants?

**4. Limitations:**
   Can you think of a situation where CLIP might confuse two different plants?
   (Hint: think about lighting, season, or photo angle.)

**5. Extension 🌟:** Try adding a photo of a *completely different* object
   (a rock, a book, a shoe). What similarity score does it get?
   Does this match your prediction?


In [ ]:
# ✏️ Type your observations here
observations = """
1. Highest similarity: ???
   Do I agree? ???

2. Embedding plot observation: ???

3. Why raw cosine is never 0: ???

4. A situation where CLIP might fail: ???

5. Extension result: ???
"""
print(observations)


---
*🌱 Well done, plant scientist! You used a state-of-the-art vision model,
cosine similarity, PCA, GPS data, and interactive maps — the same tools
real researchers use.*
